# Part 1 — Installation

*Written for idtrack 0.1.2.*

This part gets you from **"I have Python"** to **"idtrack imports, and my local repository is ready"**.

**Learning objectives**
- Install idtrack in an isolated environment (conda or venv).
- Check the installation with a short report.
- Choose your local repository: the folder of downloaded tables and graph files.
- Know how idtrack reaches Ensembl, and what to change when your network blocks a way.

> **Note:** The first graph build (Part 2) takes time and disk space. This part only checks that your environment is ready. The full installation guide, with every platform: [Installation](../installation.rst).

## 1.1 — Create an isolated environment

idtrack runs on Python 3.12 to 3.15. A fresh environment avoids dependency conflicts; either option below is fine.

**Option A: conda/mamba**

```bash
mamba create -n idtrack -c conda-forge python=3.13 -y
mamba activate idtrack
```

**Option B: venv**

```bash
python3.13 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
```

> **Python 3.15:** `h5py` publishes no wheels for 3.15 yet, so pip compiles it, which needs a C compiler and the HDF5 library with its headers installed first (e.g. `mamba install -c conda-forge hdf5`, `brew install hdf5` or `apt-get install build-essential libhdf5-dev`). Python 3.12 to 3.14 need none of this. Details: [Installation](../installation.rst).

## 1.2 — Install idtrack

idtrack is installed with pip (it is not on conda-forge), in either environment:

```bash
pip install idtrack
```

To harmonize AnnData datasets (Part 4), install the `harmonize` extra, which adds `anndata` and `scipy`:

```bash
pip install "idtrack[harmonize]"
```

The same release from GitHub, by its tag:

```bash
pip install "git+https://github.com/theislab/idtrack.git@v0.1.2"
pip install "idtrack[harmonize] @ git+https://github.com/theislab/idtrack.git@v0.1.2"
```

To work on idtrack itself (an editable install with the development tools), follow `make install` in [SETUP.md](https://github.com/theislab/idtrack/blob/main/SETUP.md).

> **Expected result:** `import idtrack` works in Python, and `idtrack.__version__` prints the version.

## 1.3 — Environment report (safe to run)

This cell prints your Python version and tries to import idtrack. (The output shown here comes from a source checkout of this version; an installed release says "an installed package".)

In [1]:
import platform
import sys
from pathlib import Path

print('Python:', sys.version.split()[0])
print('Platform:', platform.system(), platform.machine())

try:
    import idtrack

    print('idtrack version:', idtrack.__version__)
    in_site_packages = 'site-packages' in Path(idtrack.__file__).parts
    print('idtrack imported from:', 'an installed package' if in_site_packages else 'a source checkout')
    IDTRACK_OK = True
except Exception as e:
    print('idtrack import failed ->', repr(e))
    print('Fix: activate the intended environment, then run: pip install idtrack')
    IDTRACK_OK = False

Python: 3.12.16
Platform: Linux x86_64
idtrack version: 0.1.2
idtrack imported from: a source checkout


## 1.4 — Choose your local repository

idtrack keeps the **tables** it downloads from Ensembl and the **graph files** it builds in one folder of your choice, the **local repository** (and, if you make one, your own externals configuration: Part 6). You pass it to `API.build(organism, snapshot_release, local_repository)` or to the command line (`idtrack build ... --repository <folder>`).

idtrack reads no environment variable for this folder. These tutorials read it from `IDTRACK_LOCAL_REPO`, a convention of the notebooks only, so that you set it once for all of them; without it they use `./idtrack_cache`:

```bash
export IDTRACK_LOCAL_REPO=/path/to/idtrack_cache
```

> **Tip:** In a real project, put this folder in a stable place (not a temporary directory), so that tables and graphs are reused across sessions.

## 1.5 — Check that the local repository is writable (safe to run)

This cell creates the folder if needed, writes a small file and removes it again.

In [2]:
import os
from pathlib import Path

local_repo = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
local_repo.mkdir(parents=True, exist_ok=True)

test_file = local_repo / '_idtrack_write_test.txt'
test_file.write_text('ok', encoding='utf-8')
print('Local repository:', local_repo)
print('Write test:', 'OK' if test_file.read_text(encoding='utf-8') == 'ok' else 'FAILED')
test_file.unlink()

Local repository: idtrack_cache
Write test: OK


## 1.6 — How idtrack reaches Ensembl (optional check)

A first graph build needs the network; loading a graph file does not. idtrack asks **Ensembl REST** (organism names and releases) and reads **Ensembl's tables** in one of two ways, which give the same tables, rows and values:

- the public MySQL service (`ensembldb.ensembl.org`, ports 3306 and 5306, and 3337 for the human GRCh37 releases), where its ports answer;
- the MySQL dumps on `https://ftp.ensembl.org/pub/`, otherwise.

idtrack reads one environment variable, `IDTRACK_ENSEMBL_ACCESS`: unset (or `auto`), each MySQL port is tried once per process and the HTTPS dumps are used where it does not answer; `https` never tries MySQL (use it where the ports are blocked, to spare the time-outs of the tries). In Python, `idtrack.settings.configure_ensembl(access='https')` does the same. Behind a proxy, set `HTTPS_PROXY`, which `requests` honours.

The cell below asks Ensembl REST and tries each MySQL port (it changes nothing).

In [3]:
import socket

import requests

r = requests.get('https://rest.ensembl.org/info/ping', headers={'Content-Type': 'application/json'}, timeout=15)
print('Ensembl REST:', r.status_code, r.text.strip()[:80])

if IDTRACK_OK:
    import idtrack.settings
    from idtrack.build.ensembl_access import ensembl_access_mode

    print('Access mode:', ensembl_access_mode())
    host = idtrack.settings.ENSEMBL.mysql_host
    for port in idtrack.settings.ENSEMBL.mysql_port_min_release:
        try:
            with socket.create_connection((host, port), timeout=5):
                print(f'Ensembl MySQL: OK ({host}:{port})')
        except OSError as e:
            print(f'Ensembl MySQL: not reachable ({host}:{port}) -> {e.__class__.__name__}')

Ensembl REST: 200 {"ping":1}
Access mode: https


Ensembl MySQL: not reachable (ensembldb.ensembl.org:3306) -> TimeoutError


Ensembl MySQL: not reachable (ensembldb.ensembl.org:5306) -> TimeoutError


Ensembl MySQL: not reachable (ensembldb.ensembl.org:3337) -> TimeoutError


This run is on a network that blocks the MySQL ports, with `IDTRACK_ENSEMBL_ACCESS=https`: every table is read over HTTPS.

## 1.7 — What's next?

- **Part 2**: build or load a graph ([02_initialization_graph](02_initialization_graph.ipynb)).
- **Part 3**: convert identifiers and read the results ([03_api_deep_dive_human](03_api_deep_dive_human.ipynb)).
- **Part 0**, if you skipped it: the concepts ([00_idtrack_overview](00_idtrack_overview.ipynb)).

> **Milestone after Part 2:** a graph file on disk; conversions are then fast and need no network.

## 1.8 — Troubleshooting

- **`h5py` fails to install or import:** on Python 3.15, install a C compiler and HDF5 first (section 1.1); on 3.12 to 3.14, upgrade pip so that it finds the `h5py` wheel.
- **REST works, the MySQL ports do not:** common, and harmless: the tables are read over HTTPS. Set `IDTRACK_ENSEMBL_ACCESS=https` to skip the tries.
- **REST times out:** on a slow network raise the time-outs (`idtrack.settings.configure_ensembl(reading_timeout=120, rest_attempts=5)`); behind a proxy set `HTTPS_PROXY`.
- **Permission errors in the local repository:** choose a folder you can write to.
- **Slow first build:** normal. It downloads Ensembl's tables of every release (from about 12 to 14 minutes for cynomolgus over HTTPS to about 2.3 to 2.6 hours plus the download for human: [Time and memory](../performance.rst)); later runs load the graph file in seconds.